<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/5_transformer_90_model/5_3_transformer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5_3_model_transformers

## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.





## 0. Configuración del Entorno


### 0.1. Instalación de librerías


### 0.2. Importación de librerías


In [1]:
# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
import sklearn, scipy, numpy #, optuna

import joblib

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import sys, platform, lightgbm as lgb
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [2]:
print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
#print("optuna:", optuna.__version__)
#print("xgboost:", xgb.__version__)
#print("lightgbm:", lgb.__version__)

# CatBoost usa la clase para exponer versión
#print("catboost:", catboost.__version__)

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.3
sklearn: 1.6.1


### 0.3. Acceso a Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [4]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [5]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [6]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [7]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [8]:
import json
'''
# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_30 = features_dict["features_to_30"]
features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]
'''

'\n# Ruta al archivo guardado\npath = f\'{drive_path}/2_feature_engineering/features_list.json\'\n\nwith open(path, "r") as f:\n    features_dict = json.load(f)\n\n# Extraer las listas\nfeatures_to_30 = features_dict["features_to_30"]\nfeatures_to_60 = features_dict["features_to_60"]\nfeatures_to_90 = features_dict["features_to_90"]\n'

In [9]:
#print(f'Listado de features para 30min: {features_to_30}')
#print(f'Listado de features para 60min: {features_to_60}')
#print(f'Listado de features para 90min: {features_to_90}')

## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### 2.0.1. Función para cargar ventanas

In [10]:
def load_windows_and_scaler(k: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_train_sc_{k}.npz'
    path_valid  = f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_valid_sc_{k}.npz'
    path_test   = f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_test_sc_{k}.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/5_transformer_90_model/5_2_k_scaler/global_scaler.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    print(f'\tX_train_sc_{k} e y_train_{k} extraídos correctamente')
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    print(f'\tX_valid_sc_{k} e y_valid_{k} extraídos correctamente')
    X_test,  y_test  = data_test["X"],  data_test["y"]
    print(f'\tX_test_sc_{k} e y_test_{k} extraídos correctamente')

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### 2.0.2. Función para revisar información de ventanas

In [31]:
def xy_info(fold, X_train, y_train, X_valid, y_valid, X_test, y_test):
    sets = [
        ("Train", X_train, y_train),
        ("Valid", X_valid, y_valid),
        ("Test",  X_test,  y_test),
    ]

    print(f"\nResumen Fold {fold}:")
    for name, X, y in sets:
        if X is None:
            print(f"  {name}: sin datos.")
            continue

        if X.ndim == 2:
            info_dim = f"{X.shape[1]} features (aplanado)"
        else:
            info_dim = f"{X.shape[1]}×{X.shape[2]} (steps × features)"

        print(f"  {name}: {X.shape[0]} ventanas | {info_dim} | {y.shape[0]} targets")

    return X_train.shape[0], X_valid.shape[0], X_test.shape[0]

### 2.1 Carga de ventanas

In [12]:
k_folds = [ 1, 2, 3, 4, 5]

In [22]:
#Para verificar el formato de lo guardado.
#for k in k_folds:
#    print(f'Fold {k}:')
#    print('\tTrain:\t', np.load(f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_train_sc_{k}.npz').files)
#    print('\tValid:\t', np.load(f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_valid_sc_{k}.npz').files)
#    print('\tTest:\t',np.load(f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_test_sc_{k}.npz').files)

In [14]:
for k in k_folds:
    print(f'Fold {k}:')
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler = load_windows_and_scaler(k)
    # Guardar cada uno en variables dinámicas
    globals()[f"X_train_sc_{k}"] = X_train
    globals()[f"y_train_sc_{k}"] = y_train
    globals()[f"X_valid_sc_{k}"] = X_valid
    globals()[f"y_valid_sc_{k}"] = y_valid
    globals()[f"X_test_sc_{k}"]  = X_test
    globals()[f"y_test_sc_{k}"]  = y_test

Fold 1:
	X_train_sc_1 e y_train_1 extraídos correctamente
	X_valid_sc_1 e y_valid_1 extraídos correctamente
	X_test_sc_1 e y_test_1 extraídos correctamente
Fold 2:
	X_train_sc_2 e y_train_2 extraídos correctamente
	X_valid_sc_2 e y_valid_2 extraídos correctamente
	X_test_sc_2 e y_test_2 extraídos correctamente
Fold 3:
	X_train_sc_3 e y_train_3 extraídos correctamente
	X_valid_sc_3 e y_valid_3 extraídos correctamente
	X_test_sc_3 e y_test_3 extraídos correctamente
Fold 4:
	X_train_sc_4 e y_train_4 extraídos correctamente
	X_valid_sc_4 e y_valid_4 extraídos correctamente
	X_test_sc_4 e y_test_4 extraídos correctamente
Fold 5:
	X_train_sc_5 e y_train_5 extraídos correctamente
	X_valid_sc_5 e y_valid_5 extraídos correctamente
	X_test_sc_5 e y_test_5 extraídos correctamente


In [21]:
for k in k_folds:
    #print(f'Fold {k}:')
    globals()[f"X_train_sc_{k}"] = X_train
    globals()[f"y_train_sc_{k}"] = y_train
    globals()[f"X_valid_sc_{k}"] = X_valid
    globals()[f"y_valid_sc_{k}"] = y_valid
    globals()[f"X_test_sc_{k}"]  = X_test
    globals()[f"y_test_sc_{k}"]  = y_test
    xy_info(f'{k}', globals()[f"X_train_sc_{k}"], globals()[f"y_train_sc_{k}"], globals()[f"X_valid_sc_{k}"], globals()[f"y_valid_sc_{k}"], globals()[f"X_test_sc_{k}"], globals()[f"y_test_sc_{k}"])


Resumen Fold 1:
  Train: 223871 ventanas | 1080 features (aplanado) | 223871 targets
  Valid: 24898 ventanas | 1080 features (aplanado) | 24898 targets
  Test: 27852 ventanas | 1080 features (aplanado) | 27852 targets

Resumen Fold 2:
  Train: 223871 ventanas | 1080 features (aplanado) | 223871 targets
  Valid: 24898 ventanas | 1080 features (aplanado) | 24898 targets
  Test: 27852 ventanas | 1080 features (aplanado) | 27852 targets

Resumen Fold 3:
  Train: 223871 ventanas | 1080 features (aplanado) | 223871 targets
  Valid: 24898 ventanas | 1080 features (aplanado) | 24898 targets
  Test: 27852 ventanas | 1080 features (aplanado) | 27852 targets

Resumen Fold 4:
  Train: 223871 ventanas | 1080 features (aplanado) | 223871 targets
  Valid: 24898 ventanas | 1080 features (aplanado) | 24898 targets
  Test: 27852 ventanas | 1080 features (aplanado) | 27852 targets

Resumen Fold 5:
  Train: 223871 ventanas | 1080 features (aplanado) | 223871 targets
  Valid: 24898 ventanas | 1080 feature

## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [23]:
def load_metrics(data: str):
    data_path = f'{drive_path}/5_model_90_transformer/5_3_model_transformer/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [24]:
def metrics_verify(data: str) -> bool:
    data_path = f'{drive_path}/5_model_90_transformer/5_3_model_transformer/{data}.parquet'
    return os.path.exists(data_path)


In [25]:
def load_or_create_metrics (data:str):
  if metrics_verify(data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[4:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [26]:
transformers_metrics, metrics = load_or_create_metrics("5_3_transformers_metrics")

Las métricas no existen. Se crea el dataset transformers_metrics para almacenar las métricas


In [27]:
transformers_metrics

,RMSE,MAE,R2,SMAPE,DirAcc


### 3.2. Función para guardar métricas

In [28]:
def save_metrics (metrics,  metrics_name: str):
  metrics_path = f"{drive_path}/5_model_90_transformer/5_3_model_transformer/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [29]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [30]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

# Entrenamiento de Transformers

## 4. Re-formateo más Encoder mínimo

Helper para re-formatear nuestras ventanas 2D a 3D que el formato que el modelo necesita.

### 4.1. Helper: de 2D (aplanado) a 3D (B, T, F)

Lo usamos para cada set y horizonte. Nuestro `window_size = 90` y los `n_features` depende del horizonte de tiempo. El siguiente código valida que `windows_size * n_features == X.shape[1]`

In [32]:
import numpy as np
import torch
import torch.nn as nn
import math

def reshape_windows(X_flat: np.ndarray, window_size: int, n_features: int) -> np.ndarray:
    """
    Convierte X de (N, window_size * n_features) a (N, window_size, n_features).
    Valida la consistencia del producto.
    """
    assert X_flat.ndim == 2, "Se esperaba X_flat con 2D (N, T*F)."
    N, TF = X_flat.shape
    assert window_size * n_features == TF, (
        f"Inconsistencia: {window_size} * {n_features} != {TF}"
    )
    return X_flat.reshape(N, window_size, n_features)

In [33]:
window_size = 90
#features_base = ['open','high','close','low','volume']

In [40]:
features_90 = ['open',  'high',  'low',  'close',  'volume', 'ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60',  'momentum_5',  'roc_20',  'rev_mom_vol_z_60']
n_features_90 = len (features_90)
print(n_features_90)

12


In [46]:
# Re-shape de tus matrices 2D -> 3D
for k in k_folds:
    #print(f'Fold {k}:')
    globals()[f"Xtr_{k}"]  = reshape_windows(globals()[f"X_train_sc_{k}"], window_size, n_features_90)
    globals()[f"Xva_{k}"]  = reshape_windows(globals()[f"X_valid_sc_{k}"], window_size, n_features_90)
    globals()[f"Xte_{k}"]  = reshape_windows(globals()[f"X_test_sc_{k}"], window_size, n_features_90)
    print(f'Fold {k} re-shape completo')

Fold 1 re-shape completo
Fold 2 re-shape completo
Fold 3 re-shape completo
Fold 4 re-shape completo
Fold 5 re-shape completo


In [67]:
def mostrar_shapes_folds_simple(k_folds):
    for k in k_folds:
        print(f"\nShapes del Fold {k}")
        print(f"{'Set':<10}{'Escalado (.npz)':<25}{'Reshape (3D)':<25}")
        print("-" * 60)

        for nombre, s1, s2 in [
            ("Train", globals()[f'X_train_sc_{k}'].shape, globals()[f'Xtr_{k}'].shape),
            ("Valid", globals()[f'X_valid_sc_{k}'].shape, globals()[f'Xva_{k}'].shape),
            ("Test",  globals()[f'X_test_sc_{k}'].shape,  globals()[f'Xte_{k}'].shape),
        ]:
            print(f"{nombre:<10}{str(s1):<25}{str(s2):<25}")

mostrar_shapes_folds_simple(k_folds)



Shapes del Fold 1
Set       Escalado (.npz)          Reshape (3D)             
------------------------------------------------------------
Train     (223871, 1080)           (223871, 90, 12)         
Valid     (24898, 1080)            (24898, 90, 12)          
Test      (27852, 1080)            (27852, 90, 12)          

Shapes del Fold 2
Set       Escalado (.npz)          Reshape (3D)             
------------------------------------------------------------
Train     (223871, 1080)           (223871, 90, 12)         
Valid     (24898, 1080)            (24898, 90, 12)          
Test      (27852, 1080)            (27852, 90, 12)          

Shapes del Fold 3
Set       Escalado (.npz)          Reshape (3D)             
------------------------------------------------------------
Train     (223871, 1080)           (223871, 90, 12)         
Valid     (24898, 1080)            (24898, 90, 12)          
Test      (27852, 1080)            (27852, 90, 12)          

Shapes del Fold 4
Set      

### 4.2. Encoder: (backbone + posición + TransformerEncoder)

Mi TimeSeriesEncoder

- Entrada: x con shape (B, T, F)
  - B = batch size
  - T = ventana temporal (p.ej. 90 pasos)
  - F = cantidad de features por minuto

- Salida: z con shape (B, T, D)
  - D = d_model (en tu caso 128)

Es decir: para cada paso temporal devuelve un embedding de dimensión 128

In [68]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        T = x.size(1)
        return x + self.pe[:, :T]

class TimeSeriesEncoder(nn.Module):
    """
    Proyección a d_model + PositionalEncoding + TransformerEncoder (sin cabeza).
    Devuelve embeddings por paso temporal: (B, T, d_model)
    """
    def __init__(
        self,
        input_dim: int,
        d_model: int = 128,
        nhead: int = 8,
        num_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
        activation: str = "gelu",
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=activation,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)

        # init
        nn.init.xavier_uniform_(self.input_proj.weight)
        nn.init.zeros_(self.input_proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        z = self.input_proj(x)    # (B, T, D)
        z = self.pos_encoder(z)   # (B, T, D)
        z = self.encoder(z)       # (B, T, D)
        z = self.dropout(z)       # (B, T, D)
        return z

Creo un encoder por cada fold, con la misma arquitectura para todos los folds y pesos distintos (cada encoder_k es un modelo nuevo)

In [73]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Encoder por cada fold
for k in k_folds:
  globals()[f'encoder_{k}'] = TimeSeriesEncoder(
      input_dim=n_features_90,
      d_model=128,
      nhead=8,
      num_layers=2
  ).to(device)

El siguiente código es un testeo rápido para verificar que:
  - Las ventanas del fold están correctamente cargadas.
  - En encoder funciona bien.
  - Las dimensiones de salida son las esperadas.

No está entrenando nada, solo está probando.

In [90]:
def verificar_encoder (fold):
  #fold = 1  # elegir el fold
  print(f'\nFold {k}:')
  device = "cuda" if torch.cuda.is_available() else "cpu"

  # ---------- 1) Cargar las ventanas 3D del fold ----------
  # Extrae de memoria Xtr_k -> [N_train, T, F] con T = tamaño de ventana (90), y F cantidad de features (12).
  Xtr = globals()[f"Xtr_{fold}"]
  Xva = globals()[f"Xva_{fold}"]
  Xte = globals()[f"Xte_{fold}"]

  # ---------- 2) Crear tensores chicos (mini-batch) para inspección  ----------
  #Esto hace:
  # Convierte las primeras 64 ventanas a tensores PyTorch y los manda a GPU si está disponible.
  # 64 porque es un batch pequeño para inspeccionar la forma de la salida del encoder.
  xb_tr = torch.tensor(Xtr[:64], dtype=torch.float32).to(device)
  xb_va = torch.tensor(Xva[:64], dtype=torch.float32).to(device)
  xb_te = torch.tensor(Xte[:64], dtype=torch.float32).to(device)

  # ---------- 3) Encoder del fold ----------
  #Selecciona el encoder correspondiente al fold-
  enc = globals()[f"encoder_{fold}"]   # o enc_90 si es uno único

  # ---------- 4) Lista de sets ----------
  #Se arma una lista con: el mini-batch, el encoder, una etiqueta para imprimir.
  pairs = [
      (xb_tr, enc, f"Fold{fold}-train"),
      (xb_va, enc, f"Fold{fold}-valid"),
      (xb_te, enc, f"Fold{fold}-test"),
  ]

  # ---------- 5) Correr encoder ----------
  #Pasar cada mini‐batch por el encoder
  #Desactiva gradientes (no_grad()) porque no estamos entrenando. Pasa el batch por el encoder e Imprime la dimensión de la salida.

  for xb, encoder, tag in pairs:
      with torch.no_grad():
          z = encoder(xb)
      print(tag, "→", z.shape)


In [91]:
for k in k_folds:
  verificar_encoder(k)


Fold 1:
Fold1-train → torch.Size([64, 90, 128])
Fold1-valid → torch.Size([64, 90, 128])
Fold1-test → torch.Size([64, 90, 128])

Fold 2:
Fold2-train → torch.Size([64, 90, 128])
Fold2-valid → torch.Size([64, 90, 128])
Fold2-test → torch.Size([64, 90, 128])

Fold 3:
Fold3-train → torch.Size([64, 90, 128])
Fold3-valid → torch.Size([64, 90, 128])
Fold3-test → torch.Size([64, 90, 128])

Fold 4:
Fold4-train → torch.Size([64, 90, 128])
Fold4-valid → torch.Size([64, 90, 128])
Fold4-test → torch.Size([64, 90, 128])

Fold 5:
Fold5-train → torch.Size([64, 90, 128])
Fold5-valid → torch.Size([64, 90, 128])
Fold5-test → torch.Size([64, 90, 128])


Los resultados significan que:
- batch size = 64
- T = 90 pasos temporales
- d_model = 128 (dimensión del embedding por paso)

## 5. Pooling (Sin cambiar enconder)

Tenemos dos opciones simples (no requieren modificar el encoder):

- `mean`: promedio temporal.
- `last`: último paso temporal.

In [ ]:
import torch
import torch.nn as nn

class TemporalPooling(nn.Module):
    def __init__(self, mode: str = "mean"):
        super().__init__()
        assert mode in ("mean", "last"), "Soportado: 'mean' o 'last'"
        self.mode = mode

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # z: (B, T, D)
        if self.mode == "mean":
            return z.mean(dim=1)      # (B, D)
        else:  # "last"
            return z[:, -1, :]        # (B, D)

In [ ]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

for xb, enc, tag in pairs:
  with torch.no_grad():
      z = enc(xb)  # (64, 90, 128)
      p1 = pool_mean(z)      # (64, 128)
      p2 = pool_last(z)      # (64, 128)
  print(f'Para {tag}\n\tPool mean:\t{p1.shape}\tPool last:\t{p2.shape}')


## 6. Cabeza de regresión (salida escalar)

Una cabeza chiquita y estándar:

In [ ]:
class RegressionHead(nn.Module):
    def __init__(self, d_model: int = 128, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, D)
        return self.net(x).squeeze(-1)   # (B,)

In [ ]:
head_30 = RegressionHead(d_model=128, dropout=0.1)
head_60 = RegressionHead(d_model=128, dropout=0.1)
head_90 = RegressionHead(d_model=128, dropout=0.1)

Sanity check end-to-end (sin entrenar, solo shapes y un MSE “dummy”):

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

pool = TemporalPooling("mean").to(device)
head = RegressionHead(128, 0.1).to(device)

z_list = []
h_list = []
y_hat_list = []   #yhat: son las 64 predicciones del modelo (encoder→pooling→cabeza), forma (64,).

for xb, enc, tag in pairs:
  with torch.no_grad():
      z = enc(xb)     # (64, 90, 128)
      h = pool(z)               # (64, 128)
      yhat = head(h)            # (64,)
      z_list.append(z)
      h_list.append(h)
      y_hat_list.append(yhat)
  print(f'Para {tag}:\t{yhat.shape}')             # esperado: torch.Size([64])

In [ ]:
# Para verificar pérdida rápida (suponiendo y_train_30 es np.ndarray):

# Se convierte los 64 targets reales del batch a un tensor en el mismo device:
yb_ytr_30 = torch.tensor(y_train_30[:64], dtype=torch.float32).to(device)
yb_yva_30 = torch.tensor(y_valid_30[:64], dtype=torch.float32).to(device)
yb_yte_30 = torch.tensor(y_test_30[:64], dtype=torch.float32).to(device)

yb_ytr_60 = torch.tensor(y_train_60[:64], dtype=torch.float32).to(device)
yb_yva_60 = torch.tensor(y_valid_60[:64], dtype=torch.float32).to(device)
yb_yte_60 = torch.tensor(y_test_60[:64], dtype=torch.float32).to(device)

yb_ytr_90 = torch.tensor(y_train_90[:64], dtype=torch.float32).to(device)
yb_yva_90 = torch.tensor(y_valid_90[:64], dtype=torch.float32).to(device)
yb_yte_90 = torch.tensor(y_test_90[:64], dtype=torch.float32).to(device)

pairs_y = [
    (yb_ytr_30, y_hat_list[0], "30-train"),
    (yb_yva_30, y_hat_list[1], "30-valid"),
    (yb_yte_30, y_hat_list[2], "30-test"),
    (yb_ytr_60, y_hat_list[3], "60-train"),
    (yb_yva_60, y_hat_list[4], "60-valid"),
    (yb_yte_60, y_hat_list[5], "60-test"),
    (yb_ytr_90, y_hat_list[6], "90-train"),
    (yb_yva_90, y_hat_list[7], "90-valid"),
    (yb_yte_90, y_hat_list[8], "90-test"),
]

for yb, yhat, tag in pairs_y:
    # Si yhat es numpy, conviértelo a tensor primero:
    if isinstance(yhat, np.ndarray):
        yhat = torch.from_numpy(yhat)
    # Asegurar 1D y mismo device/dtype
    yhat = yhat.to(device).float().view(-1)
    yb    = yb.to(device).float().view(-1)

    #Calcula el promedio del error cuadrático del batch:
    loss = torch.nn.functional.mse_loss(yhat, yb)
    print(f'Para {tag}:\t{tuple(yhat.shape)}\tMSE={float(loss):.6f}')

Con esto validamos que el pipeline: encoder → pooling → cabeza produce un escalar por muestra y que todo coincide con `y`.

### Para evaluar todas las `y` (train/valid/test) en cada horizonte

Dos funcioneS:
1. `predict_set(...)`:

  Hace forward por lotes sobre un set completo (X_flat → reshape → encoder → pooling → cabeza) y devuelve y_pred como np.ndarray.

2. `compute_metrics(...)`:
  Calcula RMSE, MAE, R², SMAPE y Directional Accuracy.


Con `window_size` = 90:
  - Horizonte 30: total 900 → n_features_30=10
  - Horizontes 60 y 90: total 1080 → n_features_60 y 90 =12

In [ ]:
import numpy as np
import torch

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)
    eps = 1e-12
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2) + eps
    r2   = float(1 - ss_res/ss_tot)
    smape = float(np.mean(2*np.abs(y_pred - y_true)/(np.abs(y_true)+np.abs(y_pred)+eps))*100)
    diracc = float(np.mean(np.sign(y_pred) == np.sign(y_true)))
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "SMAPE": smape, "DirAcc": diracc}

@torch.no_grad()
def predict_set(enc, pool, head, X_flat: np.ndarray, window_size: int, n_features: int,
                device: str, batch_size: int = 4096) -> np.ndarray:
    # 1) reshape 2D -> 3D
    N, TF = X_flat.shape
    assert TF == window_size * n_features, f"Inconsistencia: {TF} != {window_size*n_features}"
    X = X_flat.reshape(N, window_size, n_features).astype(np.float32)

    # 2) forward en lotes
    enc.eval(); head.eval()
    preds = []
    for i in range(0, N, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device)
        z  = enc(xb)               # (B, T, D)
        h  = pool(z)               # (B, D)
        yb = head(h).cpu().numpy() # (B,)
        preds.append(yb)
    return np.concatenate(preds, axis=0)

In [ ]:
import numpy as np
import torch

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)
    eps = 1e-12
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2) + eps
    r2   = float(1 - ss_res/ss_tot)
    smape = float(np.mean(2*np.abs(y_pred - y_true)/(np.abs(y_true)+np.abs(y_pred)+eps))*100)
    diracc = float(np.mean(np.sign(y_pred) == np.sign(y_true)))
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "SMAPE": smape, "DirAcc": diracc}

@torch.no_grad()
def predict_set(enc, pool, head, X_flat: np.ndarray, window_size: int, n_features: int,
                device: str, batch_size: int = 4096) -> np.ndarray:
    # 1) reshape 2D -> 3D
    N, TF = X_flat.shape
    assert TF == window_size * n_features, f"Inconsistencia: {TF} != {window_size*n_features}"
    X = X_flat.reshape(N, window_size, n_features).astype(np.float32)

    # 2) forward en lotes
    enc.eval(); head.eval()
    preds = []
    for i in range(0, N, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device)
        z  = enc(xb)               # (B, T, D)
        h  = pool(z)               # (B, D)
        yb = head(h).cpu().numpy() # (B,)
        preds.append(yb)
    return np.concatenate(preds, axis=0)

Evaluamos train/valid/test para cada horizonte con lo que tenemos hasta el momento:
  - Encoders: enc_30, enc_60, enc_90 en device
  - Un pooling (p. ej. pool = TemporalPooling("mean").to(device))
  - Una cabeza: head = RegressionHead(128, 0.1).to(device)

Como aún no entrenamos el modelo, este paso solo servirá para probar el pipeline; los números no serán interpretados como 'buenos' o 'malos'.

In [ ]:
T = 90   # window_size usado en tus secuencias
cfg = {
    "30": {
        "n_features": 10,
        "enc": enc_30,
        "Xtr": X_train_30_scaled, "ytr": y_train_30,
        "Xva": X_valid_30_scaled, "yva": y_valid_30,
        "Xte": X_test_30_scaled,  "yte": y_test_30,
    },
    "60": {
        "n_features": 12,
        "enc": enc_60,
        "Xtr": X_train_60_scaled, "ytr": y_train_60,
        "Xva": X_valid_60_scaled, "yva": y_valid_60,
        "Xte": X_test_60_scaled,  "yte": y_test_60,
    },
    "90": {
        "n_features": 12,
        "enc": enc_90,
        "Xtr": X_train_90_scaled, "ytr": y_train_90,
        "Xva": X_valid_90_scaled, "yva": y_valid_90,
        "Xte": X_test_90_scaled,  "yte": y_test_90,
    },
}

In [ ]:
#for h, d in cfg.items():
#    ytr_pred = predict_set(d["enc"], pool, head, d["Xtr"], T, d["n_features"], device)
#    yva_pred = predict_set(d["enc"], pool, head, d["Xva"], T, d["n_features"], device)
#    yte_pred = predict_set(d["enc"], pool, head, d["Xte"], T, d["n_features"], device)
#    print(f"H{h} train:", compute_metrics(d["ytr"], ytr_pred))
#    print(f"H{h} valid:", compute_metrics(d["yva"], yva_pred))
#    print(f"H{h} test :", compute_metrics(d["yte"], yte_pred))

H30 train: {'RMSE': 0.08391394730673156, 'MAE': 0.06517232608952608, 'R2': -923.1579541105102, 'SMAPE': 187.9819069907063, 'DirAcc': 0.5212236481003891}

H30 valid: {'RMSE': 0.08685266247993059, 'MAE': 0.06756919922452623, 'R2': -703.3779285874699, 'SMAPE': 188.21201173533112, 'DirAcc': 0.5134842543363726}

H30 test : {'RMSE': 0.086167357860015, 'MAE': 0.06764863403685524, 'R2': -1280.4774812372798, 'SMAPE': 188.89294122033502, 'DirAcc': 0.5184160511944571}

H60 train: {'RMSE': 0.12328725853756824, 'MAE': 0.08989122644389953, 'R2': -994.5816693433934, 'SMAPE': 185.36909848343439, 'DirAcc': 0.5183552383364257}

H60 valid: {'RMSE': 0.12425966716858347, 'MAE': 0.09094986891948711, 'R2': -715.1700745448136, 'SMAPE': 186.4820831100463, 'DirAcc': 0.5022974956094979}

H60 test : {'RMSE': 0.12671901226524276, 'MAE': 0.09432572882647947, 'R2': -1294.026024361706, 'SMAPE': 187.43837770449636, 'DirAcc': 0.49065364351528856}

H90 train: {'RMSE': 0.15355838659818977, 'MAE': 0.1263591434614923, 'R2': -1045.766119180294, 'SMAPE': 190.8184210286885, 'DirAcc': 0.433445141017226}

H90 valid: {'RMSE': 0.15390650687511936, 'MAE': 0.12600777780045397, 'R2': -682.4927837625268, 'SMAPE': 190.44306905530874, 'DirAcc': 0.4380157336348546}

H90 test : {'RMSE': 0.15282831533449942, 'MAE': 0.12684938256658815, 'R2': -1123.2214538183036, 'SMAPE': 190.45911442538196, 'DirAcc': 0.4487213414487454}

Como resultado de esta etapa (sin entrenar), solo lo tomamos como sanity check de que:

- Todo corre sin errores.
- Las dimensiones y device están bien.
- El pipeline produce métricas sin romperse.

Después de entrenar (optimizar la cabeza junto con el encoder, o fine-tunear), esperamos que:
- MSE/RMSE baje comparado contra el baseline,
- R² suba
- SMAPE baje
- DirAcc por encima de 0.5 de forma consistente.

## 7. Preparación para Entrenamiento

### 7.1. Dataset + DataLoader (reshape dentro)

In [ ]:
import os, joblib
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# 1) Scaler de y (fit solo con y_train)
def get_y_scaler(y_train: np.ndarray, path: str = None):
    scaler = StandardScaler()
    scaler.fit(y_train.reshape(-1, 1))
    if path:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        joblib.dump(scaler, path)
    return scaler

# 2) Dataset que aplica el y_scaler
class WindowDataset(Dataset):
    def __init__(self, X_flat, y, T, F, y_scaler: StandardScaler):
        assert X_flat.shape[1] == T * F, f"Inconsistente: {X_flat.shape[1]} != {T}*{F}"
        X = X_flat.reshape(-1, T, F).astype(np.float32)
        if y_scaler is not None:
            y = y_scaler.transform(y.reshape(-1, 1)).ravel()
        self.X = X
        self.y = y.astype(np.float32)

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return torch.from_numpy(self.X[i]), torch.tensor(self.y[i], dtype=torch.float32)

# 3) Loaders genéricos (cualquier horizonte)
def make_loaders(Xtr, ytr, Xva, yva, T, F, y_scaler, bs=256, num_workers=2):
    ds_tr = WindowDataset(Xtr, ytr, T, F, y_scaler=y_scaler)
    ds_va = WindowDataset(Xva, yva, T, F, y_scaler=y_scaler)
    dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True,  pin_memory=True, num_workers=num_workers)
    dl_va = DataLoader(ds_va, batch_size=bs, shuffle=False, pin_memory=True, num_workers=num_workers)
    return dl_tr, dl_va


### 7.2. Modelo compacto (encoder + pooling mean + head)

In [ ]:
import torch.nn as nn

pool = TemporalPooling("mean").to(device)
head = RegressionHead(128, 0.1).to(device)
model_30 = enc_30  # ya creado con input_dim=10, d_model=128
model_60 = enc_60
model_90 = enc_90

### 7.3. Loop de entrenamiento (MSE, AdamW, early stopping simple)

In [ ]:
import torch
import torch.nn as nn

def train_transformer(enc: nn.Module,
                      pool: nn.Module,
                      head: nn.Module,
                      dl_tr: DataLoader,
                      dl_va: DataLoader,
                      device: str = "cuda" if torch.cuda.is_available() else "cpu",
                      lr: float = 3e-4,
                      weight_decay: float = 1e-4,
                      max_epochs: int = 50,
                      patience: int = 8,
                      grad_clip: float = 1.0,
                      use_amp: bool = True):
    enc = enc.to(device); pool = pool.to(device); head = head.to(device)
    opt = torch.optim.AdamW(list(enc.parameters()) + list(head.parameters()), lr=lr, weight_decay=weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device == "cuda"))

    best_val = float("inf"); best = None; noimp = 0
    for epoch in range(1, max_epochs + 1):
        # ---- train ----
        enc.train(); head.train()
        tr_loss = 0.0
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(use_amp and device == "cuda")):
                z = enc(xb)         # (B, T, D)
                h = pool(z)         # (B, D)
                yhat = head(h).view(-1)
                loss = nn.functional.mse_loss(yhat, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(list(enc.parameters()) + list(head.parameters()), grad_clip)
            scaler.step(opt); scaler.update()
            tr_loss += loss.item() * xb.size(0)
        tr_loss /= len(dl_tr.dataset)

        # ---- valid ----
        enc.eval(); head.eval()
        va_loss = 0.0
        with torch.no_grad():
            for xb, yb in dl_va:
                xb, yb = xb.to(device), yb.to(device)
                z = enc(xb); h = pool(z); yhat = head(h).view(-1)
                va_loss += nn.functional.mse_loss(yhat, yb).item() * xb.size(0)
        va_loss /= len(dl_va.dataset)

        print(f"Epoch {epoch:03d}  train={tr_loss:.6e}  valid={va_loss:.6e}")

        if va_loss < best_val - 1e-9:
            best_val = va_loss; noimp = 0
            best = (
                {k: v.detach().cpu().clone() for k, v in enc.state_dict().items()},
                {k: v.detach().cpu().clone() for k, v in head.state_dict().items()},
            )
        else:
            noimp += 1
            if noimp >= patience:
                print("Early stopping."); break

    if best is not None:
        enc.load_state_dict(best[0]); head.load_state_dict(best[1])

    return enc, head

In [ ]:
@torch.no_grad()
def predict_set(enc, pool, head, X_flat: np.ndarray, T: int, F: int, device: str,
                batch_size: int = 4096, y_scaler: StandardScaler = None) -> np.ndarray:
    N = X_flat.shape[0]
    X = X_flat.reshape(N, T, F).astype(np.float32)
    preds = []
    enc.eval(); head.eval()
    for i in range(0, N, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device)
        z  = enc(xb); h = pool(z); yb = head(h).cpu().numpy()
        preds.append(yb)
    y_pred_scaled = np.concatenate(preds, axis=0).reshape(-1, 1)
    if y_scaler is not None:
        y_pred = y_scaler.inverse_transform(y_pred_scaled).ravel()
    else:
        y_pred = y_pred_scaled.ravel()
    return y_pred

## 8. Entrenamiento

In [ ]:
transformers_metrics

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


### 8.1. Entrenamiento H=30 (train/valid/test)

In [ ]:
model_30_key = "transformer_30_100%"

if metrics and model_30_key in transformers_metrics.index:
    print(f"Omitimos este entrenamiento: {model_30_key} ya existe en transformers_metrics")
else:
  print(f"Entrenando modelo: {model_30_key}")
  # ---- 30 min ----
  scaler_y_30 = get_y_scaler(y_train_30)  # opcional: guarda con joblib si querés
  dl_tr_30, dl_va_30 = make_loaders(X_train_30_scaled, y_train_30,
                                    X_valid_30_scaled, y_valid_30,
                                    T=90, F=10, y_scaler=scaler_y_30, bs=256)

  enc_30, head_30 = train_transformer(enc_30, pool, head_30, dl_tr_30, dl_va_30, device=device)

  ytr_pred_30 = predict_set(enc_30, pool, head_30, X_train_30_scaled, 90, 10, device, y_scaler=scaler_y_30)
  yva_pred_30 = predict_set(enc_30, pool, head_30, X_valid_30_scaled, 90, 10, device, y_scaler=scaler_y_30)
  yte_pred_30 = predict_set(enc_30, pool, head_30, X_test_30_scaled,  90, 10, device, y_scaler=scaler_y_30)

  H30_train = compute_metrics(y_train_30, ytr_pred_30)
  H30_valid = compute_metrics(y_valid_30, yva_pred_30)
  H30_test = compute_metrics(y_test_30,  yte_pred_30)

#3min

### 8.2. Entrenamiento H=60 (train/valid/test)

In [ ]:
model_60_key = "transformer_60_100%"

if metrics and model_60_key in transformers_metrics.index:
    print(f"Omitimos este entrenamiento: {model_60_key} ya existe en transformers_metrics")
else:
  print(f"Entrenando modelo: {model_60_key}")
  scaler_y_60 = get_y_scaler(y_train_60)
  dl_tr_60, dl_va_60 = make_loaders(X_train_60_scaled, y_train_60,
                                    X_valid_60_scaled, y_valid_60,
                                    T=90, F=12, y_scaler=scaler_y_60, bs=256)

  enc_60, head_60 = train_transformer(enc_60, pool, head_60, dl_tr_60, dl_va_60, device=device)

  ytr_pred_60 = predict_set(enc_60, pool, head_60, X_train_60_scaled, 90, 12, device, y_scaler=scaler_y_60)
  yva_pred_60 = predict_set(enc_60, pool, head_60, X_valid_60_scaled, 90, 12, device, y_scaler=scaler_y_60)
  yte_pred_60 = predict_set(enc_60, pool, head_60, X_test_60_scaled,  90, 12, device, y_scaler=scaler_y_60)

  H60_train = compute_metrics(y_train_60, ytr_pred_60)
  H60_valid = compute_metrics(y_valid_60, yva_pred_60)
  H60_test = compute_metrics(y_test_60,  yte_pred_60)
#Time 4min

### 8.3. Entrenamiento H=90 (train/valid/test)

In [ ]:
model_90_key = "transformer_90_100%"

if metrics and model_90_key in transformers_metrics.index:
    print(f"Omitimos este entrenamiento: {model_90_key} ya existe en transformers_metrics")
else:
  print(f"Entrenando modelo: {model_90_key}")
  scaler_y_90 = get_y_scaler(y_train_90)
  dl_tr_90, dl_va_90 = make_loaders(X_train_90_scaled, y_train_90,
                                    X_valid_90_scaled, y_valid_90,
                                    T=90, F=12, y_scaler=scaler_y_90, bs=256)

  enc_90, head_90 = train_transformer(enc_90, pool, head_90, dl_tr_90, dl_va_90, device=device)

  ytr_pred_90 = predict_set(enc_90, pool, head_90, X_train_90_scaled, 90, 12, device, y_scaler=scaler_y_90)
  yva_pred_90 = predict_set(enc_90, pool, head_90, X_valid_90_scaled, 90, 12, device, y_scaler=scaler_y_90)
  yte_pred_90 = predict_set(enc_90, pool, head_90, X_test_90_scaled,  90, 12, device, y_scaler=scaler_y_90)

  H90_train = compute_metrics(y_train_90, ytr_pred_90)
  H90_valid = compute_metrics(y_valid_90, yva_pred_90)
  H90_test = compute_metrics(y_test_90,  yte_pred_90)

## 7. Métricas

In [ ]:
import pandas as pd

# =====================================================
# FUNCIÓN GENERAL PARA PROMEDIO PONDERADO DE MÉTRICAS
# =====================================================
def weighted_avg_metrics(metrics_dicts, weights):
    """
    Calcula el promedio ponderado de métricas (RMSE, MAE, R2, SMAPE, DirAcc)
    a partir de una lista de diccionarios de métricas y sus pesos.

    metrics_dicts : list[dict]
        Lista con métricas de train, valid, test, etc.
    weights : list[int or float]
        Lista con los pesos correspondientes (típicamente cantidad de muestras).
    """
    keys = metrics_dicts[0].keys()
    total_w = sum(weights)
    avg = {}
    for k in keys:
        avg[k] = sum(m[k] * w for m, w in zip(metrics_dicts, weights)) / total_w
    return avg




In [ ]:
# =====================================================
# CONFIGURACIÓN DE TAMAÑOS DE CADA SPLIT (misma para 30/60/90)
# =====================================================
n_train, n_valid, n_test = 193487, 41567, 41567
weights = [n_train, n_valid, n_test]




In [ ]:
# =====================================================
# CÁLCULO GLOBAL PONDERADO PARA CADA HORIZONTE
# (asumiendo que ya tienes H30_train, H30_valid, H30_test, etc.)
# =====================================================
H30_avg_w = weighted_avg_metrics([H30_train, H30_valid, H30_test], weights)
print("H30 ponderado:", H30_avg_w)


In [ ]:
H60_avg_w = weighted_avg_metrics([H60_train, H60_valid, H60_test], weights)
print("H60 ponderado:", H60_avg_w)

In [ ]:
H90_avg_w = weighted_avg_metrics([H90_train, H90_valid, H90_test], weights)
print("H90 ponderado:", H90_avg_w)

In [ ]:
transformers_metrics.loc["transformer_90_100%"] = [
    H90_avg_w["RMSE"],
    H90_avg_w["MAE"],
    H90_avg_w["R2"],
    H90_avg_w["SMAPE"],
    H90_avg_w["DirAcc"]
]

In [ ]:
transformers_metrics

In [ ]:
#H30 ponderado: {'RMSE': 0.00205251359487858, 'MAE': 0.0014152739855647042, 'R2': 0.4529695643702975, 'SMAPE': 111.80527500183028, 'DirAcc': 0.7201477834293131}
#H60 ponderado: {'RMSE': 0.0020731580412431264, 'MAE': 0.0014316376077715362, 'R2': 0.7123005464571495, 'SMAPE': 86.12493445337672, 'DirAcc': 0.8109904887915235}
#H90 ponderado: {'RMSE': 0.0022212737358381107, 'MAE': 0.0014887220190654765, 'R2': 0.7832929729590808, 'SMAPE': 77.70827785806391, 'DirAcc': 0.8419642760311039}

In [ ]:
save_metrics(transformers_metrics, "4_7_transformers_metrics")